# Observable Geometry: model audit figures

This notebook runs the shipped `distinguish` diagnostic and makes figures for the Build research article. On the project computer it uses the hash-verified **Localize-MI sub-01 model-only bundle**. On a public clone it uses a four-source **arithmetic demo**, which must never be presented as neural evidence. It reads no recorded EEG waveform or outcome array. The figures are model-conditional; they do not establish source localization accuracy, a new method, or a clinical threshold.

Run from the repository root. Install the figure extra with `python -m pip install -e ".[figures]"`. Select the `Observable Geometry (Python 3.12)` kernel if it is registered, or a Python 3.12+ environment with the dependencies.

In [ ]:
from pathlib import Path
import argparse, hashlib, json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from observable_geometry.cli import run_distinguish

ROOT = Path.cwd().resolve()
assert (ROOT / "pyproject.toml").is_file(), "Launch Jupyter from the repository root"
MODEL = ROOT / "outputs/track_b/20261029T_atomic_cli_real_model_reproduction/model_export/model_bundle.npz"
META = MODEL.with_name("model_metadata.json")
DEMO = ROOT / "examples/browser_algebra_demo.npz"
DEMO_META = ROOT / "examples/browser_algebra_demo_metadata.json"
EXPECTED_BUNDLE = "e4116465c9af32ddf4bf9e0d3a8e92428c6f1b0bce22a5db257b1c3eaf07a32f"
EXPECTED_META = "a4d52521ddb760114ba1ce740942bc3bb6232f919de62737d345beb7c48db865"
def digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
MODE = "participant_model" if MODEL.is_file() and META.is_file() else "arithmetic_demo"
bundle, metadata = (MODEL, META) if MODE == "participant_model" else (DEMO, DEMO_META)
if MODE == "participant_model":
    assert digest(bundle) == EXPECTED_BUNDLE and digest(metadata) == EXPECTED_META, "Model input hash changed"
else:
    assert DEMO.is_file() and DEMO_META.is_file(), "Public arithmetic fixture missing"
print("Mode:", MODE, "| bundle SHA-256:", digest(bundle))

## 1. Run the diagnostic

The command compares fixed-orientation forward columns after covariance-supported whitening and normalization. It excludes candidates less than 20 mm away, then saves the five highest-scoring eligible alternatives per source. The arithmetic demo uses three alternatives so every possible candidate is visible. The output directory is new on each run.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = ROOT / "outputs" / f"notebook_figures_{MODE}_{stamp}"
args = argparse.Namespace(bundle=bundle, metadata=metadata, output=OUT,
                          minimum_distance_mm=20.0, top_k=5 if MODE == "participant_model" else 3,
                          block_size=128)
run_distinguish(args)
summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
pairs = pd.read_csv(OUT / "ranked_pairs.csv")
assert len(pairs) == summary["source_count"] * args.top_k
assert np.allclose(pairs["profiled_residual_unit_snr"],
                   np.sqrt(1 - pairs["absolute_cosine"]**2), atol=1e-12)
print("Output:", OUT)
print("Sources:", summary["source_count"], "| ranked rows:", len(pairs))

## 2. Distribution of each source's best eligible alternative

Each source vertex contributes **one** score. Neighboring vertices remain spatially dependent, so the histogram is descriptive. The 0.95 marker is a reading aid, not a validated decision threshold. For the participant model, the displayed horizontal axis begins at 0.7; the arithmetic demo retains the full 0–1 range.

In [ ]:
FIG = OUT / "figures"
FIG.mkdir()
plt.rcParams.update({"font.size": 10, "figure.dpi": 120, "savefig.dpi": 300})
best = pairs.loc[pairs["rank"] == 1].copy()
fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.hist(best["absolute_cosine"], bins=32, color="#3267a8", edgecolor="white")
ax.axvline(summary["median_best_distant_abs_cosine"], color="#c45b23", lw=2,
           label=f"Median {summary['median_best_distant_abs_cosine']:.3f}")
if MODE == "participant_model":
    ax.axvline(.95, color="#333333", ls="--", lw=1.3, label="0.95 descriptive marker")
ax.set(xlabel="Best eligible absolute cosine", ylabel="Model vertices",
       title="Best alternative for each modeled source")
ax.set_xlim(0.7 if MODE == "participant_model" else 0.0, 1.0)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "figure_1_best_similarity.png")
fig.savefig(FIG / "figure_1_best_similarity.svg")
plt.show()
print("Fraction >= 0.95:", summary["fraction_best_abs_cosine_ge_0_95"])

## 3. Similarity and straight-line separation

This graph includes only the **exported top-ranked alternatives**, not all eligible source pairs. Its horizontal axis is Euclidean separation in the model coordinate frame, not distance along a folded cortical surface.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.3))
if len(pairs) > 1000:
    artist = ax.hexbin(pairs["distance_mm"], pairs["absolute_cosine"], gridsize=55,
                       mincnt=1, bins="log", cmap="viridis")
    fig.colorbar(artist, ax=ax, label="Exported pairs (log count)")
else:
    ax.scatter(pairs["distance_mm"], pairs["absolute_cosine"], s=35, alpha=.8)
ax.set(xlabel="Straight-line source separation (mm)", ylabel="Absolute cosine",
       title="Ranked alternatives after the 20 mm exclusion")
fig.tight_layout()
fig.savefig(FIG / "figure_2_distance_similarity.png")
fig.savefig(FIG / "figure_2_distance_similarity.svg")
plt.show()
print("Largest exported separation (mm):", pairs["distance_mm"].max())

## 4. Comparison with the established MNE resolution summary

This figure is available only with the verified local participant model and the separately audited B7 table. It compares two **model-derived** quantities for the same vertex IDs. A correlation does not establish better researcher decisions or localization accuracy. No p-value or confidence interval is shown because the vertices share anatomy and are not independent participants.

In [ ]:
baseline = ROOT / "outputs/track_b/20261024_b7_resolution_overlap_audit_v2/paired_source_metrics.csv"
EXPECTED_BASELINE = "f8099a98b4cf339dd8d68f56a8b85ed3c7e741d12add1501a85ba50037208bc2"
if MODE == "participant_model" and baseline.is_file():
    assert digest(baseline) == EXPECTED_BASELINE, "B7 paired table hash changed"
    b7 = pd.read_csv(baseline)
    assert len(b7) == len(best) == 8196
    merged = best[["source_index", "source_id", "absolute_cosine"]].merge(
        b7[["source_index", "vertex_id", "best_distant_abs_cosine", "peak_location_error_mm"]],
        on="source_index", validate="one_to_one")
    assert (merged["source_id"].astype(str) == merged["vertex_id"].astype(str)).all()
    assert np.allclose(merged["absolute_cosine"], merged["best_distant_abs_cosine"], atol=1e-12)
    rho = spearmanr(merged["absolute_cosine"], merged["peak_location_error_mm"]).statistic
    assert abs(rho - 0.4799332221159194) < 1e-10
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    ax.scatter(merged["absolute_cosine"], merged["peak_location_error_mm"], s=5, alpha=.16,
               rasterized=True, color="#3267a8")
    ax.set(xlabel="Best eligible forward-pattern similarity", ylabel="MNE PSF peak-location error (mm)",
           title=f"Two model diagnostics on one participant (Spearman {rho:.3f})")
    fig.tight_layout()
    fig.savefig(FIG / "figure_3_similarity_vs_psf.png")
    fig.savefig(FIG / "figure_3_similarity_vs_psf.svg")
    plt.show()
    print("Model-vertex Spearman correlation:", rho)
else:
    print("Skipped: licensed participant model and audited B7 table are needed for this comparison.")

## 5. Traceable figure manifest and suggested captions

Keep the output directory with the manuscript's working files. These figures are derived from a licensed participant model when `MODE == participant_model`; do not commit or redistribute them without assessing the dataset's terms. The public arithmetic-demo figures are only software illustrations.

In [ ]:
captions = {
 "figure_1_best_similarity": "Distribution of each model vertex's best eligible absolute cosine after a 20 mm Euclidean exclusion. One participant-specific forward model; the 0.95 line is descriptive, not a calibrated source decision threshold.",
 "figure_2_distance_similarity": "Distance and similarity of the exported top-ranked alternatives. Only ranked pairs are shown; the maximum distance in this plot does not bound all eligible pairs.",
 "figure_3_similarity_vs_psf": "Paired model-derived forward-pattern similarity and MNE point-spread peak-location error for 8,196 dependent vertices of one participant. The rank association is descriptive and does not show improved localization or decision utility."
}
manifest = {"mode": MODE, "model_bundle_sha256": digest(bundle), "metadata_sha256": digest(metadata),
            "ranked_csv_sha256": digest(OUT / "ranked_pairs.csv"), "summary_sha256": digest(OUT / "summary.json"),
            "source_count": int(summary["source_count"]), "ranked_row_count": len(pairs),
            "figures": {p.name: digest(p) for p in sorted(FIG.iterdir())},
            "captions": captions,
            "limits": "Model-conditional numerical outputs, not measured neural source recovery or a localization probability."}
(OUT / "figure_manifest.json").write_text(json.dumps(manifest, indent=2)+chr(10), encoding="utf-8")
print("Saved figures and manifest in", OUT)
print("Figures:", list(manifest["figures"]))